# Tele Marketing Interview Predictor (single notebook)

Reads Google Form applicants → computes **distance from home to placement (km)** → predicts **LANJUT / TIDAK LANJUT** with a **score and reasons** → writes everything to a Google Sheet tab.

**How to use:** fill in the settings in cell 1, then **Runtime → Run all**. Re-run the whole notebook whenever you replace the data (retraining takes about a minute).

Sections: 1 Settings · 2 Setup · 3 Geo (domicile → km) · 4 Features · 5 Labels from Rekap_All · 6 Model + reasons · 7 Train & evaluate · 8 Accuracy vs coverage · 9 Predict · 10 Write to Google Sheet

## 1. Settings (edit this cell)

In [ ]:
# ---- Where your files are in Google Drive (create this folder and put ALL files in it, incl. kec_coords.csv and regions_l12.csv) ----
BASE = "/content/drive/MyDrive/skripsi_ml"              # folder holding the files below + all outputs
FORM_XLSX = "Form Loker Tele Marketing (Jawaban) (1).xlsx"   # form responses export (training data)
REKAP_XLSX = "Rekap_All.xlsx"                           # rekap used to build labels; set None to use the form's own "Lanjut Interview" column

# ---- Training periods whose labels you trust (by form Timestamp) ----
# 2024-12..2025-11 are excluded because there is NO rekap for 2025 (those rows are "unknown", not "rejected").
# When you get more labelled data, add its period here and re-run.
RELIABLE_WINDOWS = [("2024-06-01", "2024-11-30")]
CHECK_WINDOW = ("2025-12-01", "2026-01-31")             # NOT used for training; only a sanity check on a newer period
DEFAULT_PLACEMENT = "Cisauk"                            # used when the form has no "Lokasi Penempatan" value

# ---- Google Sheet output (leave SHEET_ID empty to skip section 10) ----
SHEET_ID = ""                                           # the part of the sheet URL between /d/ and /edit
CREDS_JSON = "service_account.json"                     # service-account key, placed inside BASE
FORM_TAB = None                                         # name of the form-responses tab; None = first tab

SEED = 42

## 2. Setup

In [ ]:
!pip install -q xgboost shap rapidfuzz gspread openpyxl
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab - skipping Drive mount")
os.makedirs(BASE, exist_ok=True)
from pathlib import Path
OUT_DIR = Path(BASE) / "outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT_DIR / "model_interview.joblib"
FORM_PATH = os.path.join(BASE, FORM_XLSX); REKAP_PATH = os.path.join(BASE, REKAP_XLSX) if REKAP_XLSX else None
print("Form file exists :", os.path.exists(FORM_PATH))
print("Rekap file exists:", REKAP_PATH is None or os.path.exists(REKAP_PATH))

## 3. Geo: domicile text → kecamatan → distance in km
Needs two small reference files in your Drive folder: `kec_coords.csv` (coordinates of 7,265 kecamatan, centroid of official boundary polygons) and `regions_l12.csv` (province / regency names). Source: cahyadsn/wilayah_boundaries & wilayah (MIT licence).

In [ ]:
# Reference table of Indonesian kecamatan coordinates: two small CSV files you put in the same Drive folder (BASE).
GEO_DIR = BASE
_missing = [f for f in ("kec_coords.csv", "regions_l12.csv") if not os.path.exists(os.path.join(GEO_DIR, f))]
assert not _missing, f"Upload these files to {GEO_DIR}: {_missing}"
print("Geo reference files found in", GEO_DIR)

In [ ]:
"""Resolusi domisili (teks bebas dari Google Form) -> koordinat kecamatan -> jarak ke lokasi penempatan.

Sumber koordinat: centroid polygon kecamatan (cahyadsn/wilayah_boundaries, Kepmendagri 2025).
Jarak = garis lurus (haversine). Jarak tempuh jalan biasanya ~1.3-1.6x lebih panjang.
"""

import math
import re
import unicodedata
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz, process

DATA = Path(GEO_DIR)

# Titik lokasi penempatan. None -> pakai centroid kecamatan bernama sama (mis. "Cisauk").
# Kalau punya titik kantor yang presisi, isi (lat, lng), contoh: {"cisauk": (-6.3393, 106.6302)}
PLACEMENTS: dict[str, tuple[float, float] | None] = {"cisauk": None}

STOP = re.compile(
    r"\b(kabupaten|kab|kota|kotamadya|kecamatan|kec|kelurahan|kel|desa|provinsi|prov|"
    r"daerah khusus ibukota|dki|rt|rw|jl|jln|jalan|no|blok|perumahan|perum|komplek|kompleks|"
    r"kp|kampung|dusun|administrasi)\b"
)

ALIAS = {  # singkatan / salah ketik umum -> nama baku
    "tangsel": "tangerang selatan", "tanggerang": "tangerang", "tangerng": "tangerang",
    "tengerang": "tangerang", "jaksel": "jakarta selatan", "jakbar": "jakarta barat",
    "jakpus": "jakarta pusat", "jaktim": "jakarta timur", "jakut": "jakarta utara",
    "baten": "banten", "bante": "banten", "banyen": "banten", "banteb": "banten",
    "jabar": "jawa barat", "jawabarat": "jawa barat",
}


def norm(s) -> str:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return " ".join(ALIAS.get(w, w) for w in s.split())


def strip_admin(s: str) -> str:
    return re.sub(r"\s+", " ", STOP.sub(" ", s)).strip()


def haversine_km(lat1, lon1, lat2, lon2) -> float:
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


class GeoResolver:
    def __init__(self):
        kec = pd.read_csv(DATA / "kec_coords.csv", dtype={"kode": str})
        reg = pd.read_csv(DATA / "regions_l12.csv", dtype={"kode": str})
        kec["reg"] = kec["kode"].str[:5]
        kec["prov"] = kec["kode"].str[:2]
        kec["name_n"] = kec["nama"].map(lambda x: strip_admin(norm(x)))
        names = dict(zip(reg["kode"], reg["nama"]))
        kec["reg_name"] = kec["reg"].map(names)
        self.kec = kec.reset_index(drop=True)
        self._kec_names = self.kec["name_n"].tolist()
        self._kec_reg = self.kec["reg"].tolist()
        self._kec_prov = self.kec["prov"].tolist()
        self._kec_rows = self.kec.to_dict("records")

        regs = reg[reg["kode"].str.len() == 5].copy()
        regs["is_kota"] = regs["nama"].str.lower().str.startswith("kota")
        regs["name_n"] = regs["nama"].map(lambda x: strip_admin(norm(x)))
        regs = regs.join(kec.groupby("reg")[["lat", "lng"]].mean(), on="kode").dropna(subset=["lat"])
        self.regs = regs.reset_index(drop=True)
        self._reg_rows = list(zip(self.regs["kode"], self.regs["name_n"], self.regs["is_kota"]))
        self._reg_by_code = {r["kode"]: r for r in self.regs.to_dict("records")}
        provs = reg[reg["kode"].str.len() == 2]
        self._prov_rows = [(k, strip_admin(norm(n))) for k, n in zip(provs["kode"], provs["nama"])]
        self._admin_names = {n for _k, n, _c in self._reg_rows} | {n for _k, n in self._prov_rows}
        self._cache_reg: dict = {}
        self._cache_res: dict = {}

    # ---------- kab/kota ----------
    def _regency_candidates(self, text_n: str, kota_hint):
        key = (text_n, kota_hint)
        if key in self._cache_reg:
            return self._cache_reg[key]
        t = strip_admin(text_n)
        out: dict[str, float] = {}
        if t:
            padded = f" {t} "
            for kode, n, is_kota in self._reg_rows:
                if t == n:
                    s = 100.0
                elif f" {n} " in padded:
                    s = 92.0
                else:
                    s = fuzz.ratio(t, n)
                    if s < 88:
                        continue
                if kota_hint is not None and is_kota != kota_hint:
                    s -= 6  # "Tangerang" polos tetap menerima Kota maupun Kabupaten
                out[kode] = max(out.get(kode, 0), s)
            if t in ("jakarta", "jakarta kota"):  # "jakarta" polos -> semua kota di DKI
                for kode, _n, _k in self._reg_rows:
                    if kode.startswith("31"):
                        out[kode] = max(out.get(kode, 0), 80.0)
        self._cache_reg[key] = out
        return out

    # ---------- utama ----------
    def resolve(self, prov="", kab="", kec="", kel="", alamat=""):
        key = tuple(map(norm, (prov, kab, kec, kel, alamat)))
        if key not in self._cache_res:
            self._cache_res[key] = self._resolve(*key)
        return self._cache_res[key]

    def _resolve(self, prov_n, kab_n, kec_n, kel_n, alm_n):
        kota_hint = True if re.search(r"\bkota\b", kab_n) else False if re.search(r"\bkab(upaten)?\b", kab_n) else None

        reg_scores: dict[str, float] = {}
        for txt, w, hint in ((kab_n, 1.0, kota_hint), (prov_n, 0.8, None), (kec_n, 0.5, None),
                             (kel_n, 0.4, None), (alm_n, 0.4, None)):
            for k, s in self._regency_candidates(txt, hint).items():
                reg_scores[k] = max(reg_scores.get(k, 0), s * w)

        prov_codes = set()
        for kode, pn in self._prov_rows:
            for txt in (prov_n, kab_n):
                t = strip_admin(txt)
                if t and (t == pn or (len(t) > 3 and fuzz.ratio(t, pn) >= 88)):
                    prov_codes.add(kode)
        if re.search(r"\bdki\b|jakarta", prov_n + " " + kab_n):
            prov_codes.add("31")

        keep: set[str] = set()
        if reg_scores:
            top = max(reg_scores.values())
            keep = {k for k, s in reg_scores.items() if s >= top - 15 and s >= 60}
        if keep:  # cari di seluruh provinsi terkait; kecamatan di luar kab/kota yang disebut hanya boleh lewat kolom kecamatan
            provs_keep = {k[:2] for k in keep}
            idx = [i for i, p in enumerate(self._kec_prov) if p in provs_keep]
        elif prov_codes:
            idx = [i for i, p in enumerate(self._kec_prov) if p in prov_codes]
        else:
            idx = list(range(len(self._kec_names)))
        strict = not keep  # tanpa petunjuk kab/kota -> wajib cocok kuat
        names = [self._kec_names[i] for i in idx]

        best = None  # (skor, row, field)
        for field_name, txt, w in (("kecamatan", kec_n, 1.0), ("kelurahan", kel_n, 0.9), ("alamat", alm_n, 0.85),
                                   ("kabupaten", kab_n, 0.8), ("provinsi", prov_n, 0.8)):
            t = strip_admin(txt)
            if len(t) < 3:
                continue
            padded = f" {t} "
            cand: dict[int, float] = {}
            skip_admin = field_name != "kecamatan"  # nama kab/kota di kolom lain bukan nama kecamatan
            for j, n in enumerate(names):
                if len(n) < 3 or (skip_admin and n in self._admin_names):
                    continue
                if t == n:
                    cand[j] = 100.0
                elif len(n) >= 5 and f" {n} " in padded:
                    cand[j] = 93.0
            if field_name in ("kecamatan", "kelurahan"):
                for _n, sc, j in process.extract(t, names, scorer=fuzz.ratio, score_cutoff=86, limit=8):
                    if not (skip_admin and names[j] in self._admin_names):
                        cand[j] = max(cand.get(j, 0), sc)
            for j, sc in cand.items():
                r = self._kec_rows[idx[j]]
                outside = bool(keep) and r["reg"] not in keep
                if outside and not (field_name == "kecamatan" and sc >= 93):
                    continue
                tot = sc * w + reg_scores.get(r["reg"], 0) * 0.05 - (10 if outside else 0)
                if field_name == "kecamatan" and norm(r["nama"]) == txt:  # nama persis (mis. "Bangkinang" vs "Bangkinang Kota")
                    tot += 0.5
                if best is None or tot > best[0]:
                    best = (tot, r, field_name, sc)

        if best and (best[3] >= 93 if strict else best[0] >= 74):
            r = best[1]
            return dict(lat=r["lat"], lng=r["lng"], level="kecamatan",
                        match=f"{r['nama']}, {r['reg_name']}", conf=round(min(best[0], 100) / 100, 2))
        if reg_scores:
            k = max(reg_scores, key=reg_scores.get)
            if reg_scores[k] >= 80:
                r = self._reg_by_code[k]
                return dict(lat=r["lat"], lng=r["lng"], level="kabupaten_kota", match=r["nama"],
                            conf=round(reg_scores[k] / 100, 2))
        return dict(lat=float("nan"), lng=float("nan"), level="tidak_terbaca", match="", conf=0.0)

    # ---------- penempatan & jarak ----------
    def placement_coord(self, name: str):
        key = norm(name)
        if PLACEMENTS.get(key):
            return PLACEMENTS[key]
        r = [row for row in self._kec_rows if row["name_n"] == strip_admin(key)]
        if not r:
            raise ValueError(f"Lokasi penempatan '{name}' tidak ditemukan. Tambahkan koordinat di PLACEMENTS.")
        pri = {"36": 0, "32": 1, "31": 2}
        r.sort(key=lambda x: pri.get(x["prov"], 9))
        return float(r[0]["lat"]), float(r[0]["lng"])

    def distance_km(self, lat, lng, placement: str):
        if lat is None or lng is None or (isinstance(lat, float) and math.isnan(lat)):
            return float("nan")
        plat, plng = self.placement_coord(placement)
        return haversine_km(lat, lng, plat, plng)

## 4. Features (cleaning + feature engineering, identical for training and prediction)

In [ ]:
"""Cleaning + feature engineering. Dipakai SAMA PERSIS untuk training dan prediksi (anti train/serve skew)."""

import re

import numpy as np
import pandas as pd


# ---- pemetaan header Google Form -> nama pendek (cocok by prefix, tahan terhadap spasi/typo akhir) ----
HEADER_MAP = [
    ("timestamp", "ts"), ("nama lengkap", "nama"), ("nomor ktp", "ktp"), ("nomor hp", "hp"),
    ("alamat email", "email"), ("tanggal lahir", "tgl_lahir"), ("jenis kelamin", "jk"),
    ("status pernikahan", "nikah"), ("provinsi tinggal", "prov"), ("kabupaten", "kab"),
    ("kecamatan tinggal", "kec"), ("kelurahan tinggal", "kel"), ("alamat lengkap", "alamat"),
    ("pendidikan terakhir", "pend"), ("jurusan", "jurusan"), ("nama sekolah", "sekolah"),
    ("nilai terakhir", "nilai"), ("nama perusahaan", "perusahaan"), ("bidang perusahaan", "bidang"),
    ("pengalaman kerja 3 thn terakhir di bagian", "bagian"), ("pengalaman kerja 3 thn terakhir di level", "level"),
    ("berapa lama bekerja di pengalaman terakhir", "lama_kerja_terakhir"),
    ("jelaskan sekilas", "tanggung_jawab"), ("mengetahui loker", "sumber"), ("lampirkan curriculum", "cv"),
    ("dari semua pengalaman kerja anda berapa durasi", "durasi_terlama"),
    ("dari semua pengalaman kerja anda yang paling lama", "jabatan_terlama"),
    ("apakah pernah memiliki pengalaman sales", "pernah_sales"),
    ("berpengalaman sebagai sales", "lama_sales"),
    ("apakah pernah memiliki pengalaman crew", "pernah_crew"),
    ("berpengalaman sebagai crew", "lama_crew"),
    ("alasan mengapa", "alasan"), ("kapan anda terakhir bekerja", "terakhir_kerja"),
    ("apakah saat ini dalam kondisi hamil", "hamil"), ("apakah anda sedang berkuliah", "kuliah"),
    ("apakah anda memiliki rencana", "rencana_kuliah"), ("tempat lahir", "tempat_lahir"),
    ("lokasi penempatan", "penempatan"), ("lanjut interview", "label"),
]

# ---- fitur yang masuk model ----
NUM_FEATURES = [
    "jarak_km", "usia", "nilai_norm", "lama_sales_ord", "lama_crew_ord", "lama_kerja_ord", "durasi_terlama_ord",
]
BIN_FEATURES = [
    "menikah", "punya_anak", "pernah_sales", "pernah_crew", "kuliah_sekarang", "rencana_kuliah",
    "jabatan_sales_like", "jabatan_tele_like", "jabatan_collection_like", "lokasi_presisi", "dekat_10km",
]
CAT_FEATURES = ["pend_level", "bidang", "bagian", "level", "sumber", "terakhir_kerja"]
FEATURES = NUM_FEATURES + BIN_FEATURES + CAT_FEATURES

# Dikecualikan SENGAJA dari model: jenis kelamin, kehamilan, tempat lahir, nama/KTP/HP/email (identitas / diskriminatif).
# Usia & status pernikahan DIPAKAI karena merupakan persyaratan sebagian posisi PT XYZ (keputusan peneliti).


def _norm_header(h) -> str:
    return re.sub(r"\s+", " ", str(h)).strip().lower()


def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    new, used = {}, set()
    for col in df.columns:
        h = _norm_header(col)
        for prefix, short in HEADER_MAP:
            if h.startswith(prefix) and short not in used:
                new[col] = short
                used.add(short)
                break
    return df.rename(columns=new)


def _s(x) -> str:
    return "" if pd.isna(x) else re.sub(r"\s+", " ", str(x)).strip()


def _yn(x) -> float:
    v = _s(x).lower()
    return 1.0 if v.startswith("ya") else 0.0 if v.startswith("tidak") else np.nan


def _ord_years(x) -> float:
    v = _s(x).lower().replace(" ", "")
    if not v:
        return np.nan
    if "fresh" in v:
        return 0.0
    if v.startswith("<1"):
        return 1.0
    if v.startswith("1-3"):
        return 2.0
    if v.startswith(">2") or v.startswith(">3"):
        return 3.0
    return np.nan


def parse_nilai(x) -> float:
    """Nilai/IPK campur skala (4, 10, 100) -> skala 0-100."""
    v = _s(x).replace(",", ".")
    m = re.search(r"\d+(?:\.\d+)?", v)
    if not m:
        return np.nan
    f = float(m.group())
    if f <= 0:
        return np.nan
    if f <= 4.0:
        return f / 4 * 100
    if f <= 10:
        return f * 10
    return f if f <= 100 else np.nan


def pend_level(x) -> str:
    v = _s(x).lower()
    for key, lab in (("s2", "S2"), ("s1", "S1"), ("d4", "D4"), ("d3", "D3"), ("d1", "D1"), ("d2", "D2")):
        if re.search(rf"\b{key}\b", v):
            return lab
    if "kuliah" in v or "mahasiswa" in v:
        return "Sedang Kuliah"
    if "sma" in v or "smk" in v or "ma" == v:
        return "SMA/SMK"
    if "smp" in v:
        return "SMP"
    return "Lainnya"


def _kw(x, words) -> float:
    v = _s(x).lower()
    return float(any(w in v for w in words))


def build_features(raw: pd.DataFrame, geo: GeoResolver | None = None, placement_default: str = "Cisauk") -> pd.DataFrame:
    df = rename_columns(raw).copy()
    missing = [short for _p, short in HEADER_MAP if short not in df.columns and short != "label"]
    if missing:  # header form berubah / kolom hilang -> jangan crash, isi kosong (model tetap jalan dgn imputasi)
        import warnings
        warnings.warn(f"Kolom form tidak ditemukan, diisi kosong: {missing}")
        for m in missing:
            df[m] = np.nan
    geo = geo or GeoResolver()
    out = pd.DataFrame(index=df.index)

    ts = pd.to_datetime(df.get("ts"), errors="coerce", dayfirst=True)
    lahir = pd.to_datetime(df.get("tgl_lahir"), errors="coerce", dayfirst=True)
    usia = ((ts.fillna(pd.Timestamp.today()) - lahir).dt.days / 365.25)
    out["usia"] = usia.where(usia.between(16, 65))

    nikah = df["nikah"].map(_s).str.lower()
    out["menikah"] = nikah.str.startswith(("menikah", "cerai")).astype(float)
    out["punya_anak"] = nikah.str.contains("memiliki anak") & ~nikah.str.contains("belum")
    out["punya_anak"] = out["punya_anak"].astype(float)

    # ---- domisili -> jarak ----
    geo_cols = ["prov", "kab", "kec", "kel", "alamat"]
    res = [geo.resolve(*r) for r in df.reindex(columns=geo_cols).itertuples(index=False, name=None)]
    g = pd.DataFrame(res, index=df.index)
    plc = df["penempatan"].map(_s).replace("", placement_default) if "penempatan" in df else placement_default
    plc = plc if isinstance(plc, pd.Series) else pd.Series(plc, index=df.index)
    out["jarak_km"] = [geo.distance_km(a, b, p) for a, b, p in zip(g["lat"], g["lng"], plc)]
    out["lokasi_presisi"] = (g["level"] == "kecamatan").astype(float)
    out["dekat_10km"] = (out["jarak_km"] <= 10).astype(float)
    out.loc[out["jarak_km"].isna(), "dekat_10km"] = np.nan
    out["geo_match"], out["geo_level"] = g["match"], g["level"]

    out["pend_level"] = df["pend"].map(pend_level)
    out["nilai_norm"] = df["nilai"].map(parse_nilai)
    for c in ("bidang", "bagian", "level", "terakhir_kerja"):
        out[c] = df[c].map(_s).replace("", "Tidak diisi")
    src = df["sumber"].map(_s).str.lower().replace("", "tidak diisi")
    out["sumber"] = src.where(src.isin(["rekan", "kita lulus", "glints", "pintarnya", "instagram"]), "lainnya")

    out["lama_sales_ord"] = df["lama_sales"].map(_ord_years)
    out["lama_crew_ord"] = df["lama_crew"].map(_ord_years)
    out["lama_kerja_ord"] = df["lama_kerja_terakhir"].map(_ord_years)
    out["durasi_terlama_ord"] = df["durasi_terlama"].map(_ord_years)
    out["pernah_sales"] = df["pernah_sales"].map(_yn)
    out["pernah_crew"] = df["pernah_crew"].map(_yn)
    # lama sales/crew tidak bermakna kalau jawaban "Tidak" (form tetap mengisi default "< 1 tahun")
    out.loc[out["pernah_sales"] == 0, "lama_sales_ord"] = 0.0
    out.loc[out["pernah_crew"] == 0, "lama_crew_ord"] = 0.0
    out["kuliah_sekarang"] = df["kuliah"].map(_yn)
    out["rencana_kuliah"] = df["rencana_kuliah"].map(lambda x: 1.0 if _s(x).lower().startswith(("ya", "sedang")) else 0.0)

    jab = df["jabatan_terlama"].map(_s) + " " + df["bagian"].map(_s)
    out["jabatan_sales_like"] = jab.map(lambda x: _kw(x, ["sales", "marketing", "sale ", "spg", "spb", "canvass", "promotor", "account officer"]))
    out["jabatan_tele_like"] = jab.map(lambda x: _kw(x, ["tele", "call center", "customer service", "cs ", "csr", "contact center"]))
    out["jabatan_collection_like"] = jab.map(lambda x: _kw(x, ["collect", "penagih", "remedial"]))

    # meta (bukan fitur)
    for c in ("ts", "nama", "ktp", "hp", "email", "label"):
        if c in df:
            out["_" + c] = df[c]
    return out


def to_label(s: pd.Series) -> pd.Series:
    return s.map(lambda x: 1 if str(x).strip().lower() in ("true", "1", "ya", "yes", "lolos") else 0)


def dedupe_applicants(feat: pd.DataFrame) -> pd.DataFrame:
    """Satu baris per pelamar (KTP). Ambil submit terakhir; label = 1 jika pernah lolos pada submit mana pun."""
    key = feat["_ktp"].map(lambda x: re.sub(r"\D", "", str(x)))
    key = key.where(key.str.len() >= 10, feat["_nama"].map(_s).str.lower() + "|" + feat["_hp"].map(lambda x: re.sub(r"\D", "", str(x))))
    f = feat.assign(_key=key, _y=to_label(feat["_label"]))
    f = f.sort_values("_ts")
    ymax = f.groupby("_key")["_y"].max()
    last = f.groupby("_key", as_index=False).tail(1).copy()
    last["_y"] = last["_key"].map(ymax)
    return last.sort_values("_ts").reset_index(drop=True)

## 5. Labels from Rekap_All (match by phone number + name)

In [ ]:
"""Label 'tembus ke interview' dari Rekap_All: cocok lewat No HP (utama) atau nama persis.

Kenapa bukan nama saja: nama sering beda ejaan antara form dan rekap ("Nurlarla" vs "Nurlaela"), No HP lebih stabil.
Untuk menghindari HP salah-ketik / nomor dipakai bersama, kecocokan HP wajib disertai kemiripan nama >= 60.
"""

import re
import warnings

import pandas as pd
from rapidfuzz import fuzz

UNITS = None  # None = semua unit rekap (lolos seleksi berkas ke unit mana pun = tembus); contoh filter: ("tele marketing",)


def norm_hp(x) -> str:
    s = str(x)
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    d = re.sub(r"\D", "", s)
    if d.startswith("62"):
        d = d[2:]
    d = d.lstrip("0")
    return d if 8 <= len(d) <= 13 else ""


def norm_name(x) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", str(x).lower())).strip()


def load_rekap(path, units=UNITS) -> pd.DataFrame:
    warnings.filterwarnings("ignore")
    x = pd.ExcelFile(path)
    parts = []
    kf = x.parse("Kandidat Fix")
    parts.append(pd.DataFrame({"nama": kf["Nama"], "hp": kf["No HP"], "unit": kf["Unit"]}))
    if "Kandidat" in x.sheet_names:
        kd = x.parse("Kandidat")
        parts.append(pd.DataFrame({"nama": kd["Nama Kandidat"], "hp": None, "unit": kd["Unit"]}))
    rek = pd.concat(parts, ignore_index=True)
    rek = rek[rek["nama"].notna()]
    if units:
        rek = rek[rek["unit"].astype(str).str.strip().str.lower().isin(units)]
    rek["n"] = rek["nama"].map(norm_name)
    rek["hp"] = rek["hp"].map(norm_hp)
    return rek[rek["n"].str.len() > 2]


def relabel(raw: pd.DataFrame, rekap_path, units=UNITS) -> pd.Series:
    """1 jika pelamar ada di rekap (HP cocok + nama mirip, atau nama persis), else 0. Index = index raw."""
    rek = load_rekap(rekap_path, units)
    names = set(rek["n"])
    by_hp: dict[str, list[str]] = {}
    for h, n in zip(rek["hp"], rek["n"]):
        if h:
            by_hp.setdefault(h, []).append(n)
    cols = {re.sub(r"\s+", " ", str(c)).strip().lower(): c for c in raw.columns}
    c_nama = next(c for k, c in cols.items() if k.startswith("nama lengkap"))
    c_hp = next(c for k, c in cols.items() if k.startswith("nomor hp"))
    out = []
    for nama, hp in zip(raw[c_nama], raw[c_hp]):
        n, h = norm_name(nama), norm_hp(hp)
        ok = n in names or (h in by_hp and max(fuzz.token_sort_ratio(n, m) for m in by_hp[h]) >= 60)
        out.append(int(ok))
    return pd.Series(out, index=raw.index, name="label")

## 6. Model + reasons
Three models are compared (Logistic Regression, Random Forest, XGBoost). Reasons come from SHAP contributions, grouped into themes and phrased as sentences with historical pass rates.

In [ ]:
"""Pipeline model + penjelasan (alasan) berbahasa Indonesia."""

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


JARAK_CAP_KM = 100.0  # jarak > 100 km dianggap sama (outlier pelamar luar pulau)


def prepare_X(feat: pd.DataFrame) -> pd.DataFrame:
    X = feat[FEATURES].copy()
    X["jarak_km"] = X["jarak_km"].clip(upper=JARAK_CAP_KM)
    for c in CAT_FEATURES:
        X[c] = X[c].astype(str)
    return X


def make_pipeline(name: str, seed: int = 42) -> Pipeline:
    scale = name == "logreg"
    num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if scale else [])
    pre = ColumnTransformer(
        [
            ("num", Pipeline([(f"s{i}", t) for i, t in enumerate(num)]), NUM_FEATURES),
            ("bin", SimpleImputer(strategy="constant", fill_value=0.0), BIN_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=8, sparse_output=False), CAT_FEATURES),
        ],
        verbose_feature_names_out=True,
    )
    if name == "logreg":
        clf = LogisticRegression(C=0.3, max_iter=2000, class_weight="balanced", random_state=seed)
    elif name == "rf":
        clf = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, max_features="sqrt",
                                     class_weight="balanced_subsample", n_jobs=-1, random_state=seed)
    elif name == "xgb":
        from xgboost import XGBClassifier  # import lazy: app web tidak butuh xgboost

        clf = XGBClassifier(n_estimators=250, max_depth=3, learning_rate=0.04, subsample=0.8, colsample_bytree=0.7,
                            min_child_weight=3, reg_lambda=3.0, eval_metric="logloss", n_jobs=4, random_state=seed)
    else:
        raise ValueError(name)
    return Pipeline([("pre", pre), ("clf", clf)])


# ----------------------------------------------------------------------------------------------
# Penjelasan: SHAP per fitur -> dikelompokkan per tema -> kalimat Indonesia
# ----------------------------------------------------------------------------------------------
GROUPS = {
    "jarak": ["jarak_km", "dekat_10km", "lokasi_presisi"],
    "usia": ["usia"],
    "pendidikan": ["pend_level", "nilai_norm"],
    "sales": ["pernah_sales", "lama_sales_ord", "jabatan_sales_like"],
    "tele": ["jabatan_tele_like", "bagian"],
    "collection": ["jabatan_collection_like"],
    "crew": ["pernah_crew", "lama_crew_ord"],
    "pengalaman": ["lama_kerja_ord", "durasi_terlama_ord", "level", "bidang"],
    "status_kerja": ["terakhir_kerja"],
    "kuliah": ["kuliah_sekarang", "rencana_kuliah"],
    "keluarga": ["menikah", "punya_anak"],
    "sumber": ["sumber"],
}
FEATURE_TO_GROUP = {f: g for g, fs in GROUPS.items() for f in fs}


def _orig_feature(transformed_name: str) -> str:
    n = transformed_name.split("__", 1)[1]
    for f in sorted(FEATURES, key=len, reverse=True):
        if n == f or n.startswith(f + "_"):
            return f
    return n


class Explainer:
    """Kontribusi per fitur. Untuk Logistic Regression: coef * (x - rata-rata training) (identik dgn SHAP linear);
    untuk model pohon: SHAP TreeExplainer. State-nya berupa dict sederhana (bukan objek kustom), jadi artefak model
    bisa di-load dari notebook, Streamlit, atau script mana pun tanpa mengimpor kelas ini saat menyimpan."""

    def __init__(self, pipe: Pipeline, state: dict | None = None, background_X: pd.DataFrame | None = None):
        self.pipe = pipe
        self.names = list(pipe["pre"].get_feature_names_out())
        self.orig = [_orig_feature(n) for n in self.names]
        clf = pipe["clf"]
        self.linear = isinstance(clf, LogisticRegression)
        if self.linear:
            if state is None:
                state = {"kind": "linear", "mu": pipe["pre"].transform(background_X).mean(axis=0)}
            self.mu = np.asarray(state["mu"], dtype=float)
            self.coef = clf.coef_[0]
        else:
            state = {"kind": "tree"}
            import shap  # import lazy: hanya dibutuhkan untuk model pohon

            self.expl = shap.TreeExplainer(clf)
        self.state = state

    def to_state(self) -> dict:
        return {k: (np.asarray(v) if k == "mu" else v) for k, v in self.state.items()}

    def contributions(self, X: pd.DataFrame) -> pd.DataFrame:
        """Kontribusi per grup tema (log-odds). Positif = mendorong LANJUT."""
        Z = self.pipe["pre"].transform(X)
        if self.linear:
            sv = (np.asarray(Z, dtype=float) - self.mu) * self.coef
        else:
            sv = self.expl.shap_values(Z)
            if isinstance(sv, list):  # RF versi lama
                sv = sv[1]
            sv = np.asarray(sv)
            if sv.ndim == 3:  # RF versi baru: (n, f, classes)
                sv = sv[:, :, 1]
        df = pd.DataFrame(sv, columns=self.orig, index=X.index)
        df = df.T.groupby(level=0).sum().T
        out = pd.DataFrame(index=X.index)
        for g, fs in GROUPS.items():
            out[g] = df[[f for f in fs if f in df.columns]].sum(axis=1)
        return out


JARAK_BINS = [-0.1, 5, 10, 15, 20, 30, 60, 1e9]
JARAK_LABELS = ["≤5 km", "5–10 km", "10–15 km", "15–20 km", "20–30 km", "30–60 km", ">60 km"]


def compute_rates(feat: pd.DataFrame, y: pd.Series, min_n: int = 8) -> dict:
    """Tingkat lolos historis per kategori (untuk kalimat alasan yang berbasis data)."""
    rates = {"_avg": float(y.mean())}
    for c in ("pend_level", "sumber", "terakhir_kerja", "bagian", "bidang"):
        g = pd.DataFrame({"v": feat[c].astype(str), "y": y.values}).groupby("v")["y"].agg(["mean", "size"])
        rates[c] = {k: (float(m), int(n)) for k, (m, n) in g.iterrows() if n >= min_n}
    b = pd.cut(feat["jarak_km"].clip(upper=1e8), JARAK_BINS, labels=JARAK_LABELS)
    g = pd.DataFrame({"v": b.astype(str), "y": y.values}).groupby("v")["y"].agg(["mean", "size"])
    rates["jarak"] = {k: (float(m), int(n)) for k, (m, n) in g.iterrows() if n >= min_n}
    return rates


def _hist(rates, key, value) -> str:
    if not rates or key not in rates or value not in rates[key]:
        return ""
    m, n = rates[key][value]
    return f" (historis {m:.0%} lolos dari {n} pelamar; rata-rata {rates['_avg']:.0%})"


def _jarak_bucket(d):
    return str(pd.cut([min(d, 1e8)], JARAK_BINS, labels=JARAK_LABELS)[0])


def _jarak_txt(r):
    d = r["jarak_km"]
    return "jarak domisili tidak terbaca" if pd.isna(d) else f"±{d:.0f} km"


def _lama(v):
    return {0: "tanpa pengalaman", 1: "<1 tahun", 2: "1–3 tahun", 3: ">3 tahun"}.get(v, "-")


def reason_text(group: str, positive: bool, r: pd.Series, placement: str, rates: dict | None = None) -> str:
    """Satu kalimat alasan untuk tema `group`, sesuai arah pengaruhnya."""
    if group == "jarak":
        d = r["jarak_km"]
        if pd.isna(d):
            return "Domisili tidak terbaca sehingga jarak ke lokasi penempatan tidak bisa dihitung"
        h = _hist(rates, "jarak", _jarak_bucket(d))
        return (f"Domisili dekat dengan lokasi penempatan {placement}, {_jarak_txt(r)}{h}" if positive
                else f"Domisili cukup jauh dari lokasi penempatan {placement}, {_jarak_txt(r)}{h}")
    if group == "usia":
        u = r["usia"]
        return (f"Usia {u:.0f} tahun berada pada rentang yang umumnya lolos" if positive
                else f"Usia {u:.0f} tahun berada di luar rentang yang umumnya lolos") if not pd.isna(u) else "Usia tidak terbaca"
    if group == "pendidikan":
        n = r["nilai_norm"]
        extra = "" if pd.isna(n) else f", nilai ≈{n:.0f}/100"
        h = _hist(rates, "pend_level", r["pend_level"])
        return (f"Pendidikan {r['pend_level']}{extra} sesuai profil yang umumnya lolos{h}" if positive
                else f"Pendidikan {r['pend_level']}{extra} kurang sesuai profil yang umumnya lolos{h}")
    if group == "sales":
        if positive:
            return f"Punya pengalaman sales/marketing ({_lama(r['lama_sales_ord'])})"
        return "Belum/minim pengalaman sales atau marketing" if r["pernah_sales"] != 1 or r["lama_sales_ord"] <= 1 \
            else "Latar belakang sales kurang menonjol dibanding pelamar yang lolos"
    if group == "tele":
        h = _hist(rates, "bagian", r["bagian"])
        return (f"Pengalaman terakhir di bagian {r['bagian']} relevan dengan telemarketing{h}" if positive
                else f"Pengalaman terakhir ({r['bagian']}) kurang relevan dengan telemarketing{h}")
    if group == "collection":
        return "Punya latar belakang penagihan/collection" if positive else "Tidak punya latar belakang penagihan/collection"
    if group == "crew":
        return "Pengalaman crew store dinilai menunjang" if positive else "Pengalaman crew store kurang menunjang untuk posisi ini"
    if group == "pengalaman":
        h = _hist(rates, "bidang", r["bidang"])
        return (f"Riwayat kerja ({r['bidang']}, level {r['level']}) mendukung{h}" if positive
                else f"Riwayat kerja ({r['bidang']}, level {r['level']}) kurang mendukung{h}")
    if group == "status_kerja":
        h = _hist(rates, "terakhir_kerja", r["terakhir_kerja"])
        return (f"Status kerja terakhir ({r['terakhir_kerja']}) mendukung{h}" if positive
                else f"Status kerja terakhir ({r['terakhir_kerja']}) menurunkan peluang{h}")
    if group == "kuliah":
        return "Status/rencana kuliah tidak menjadi hambatan" if positive else "Sedang atau berencana kuliah, berpotensi mengganggu komitmen kerja"
    if group == "keluarga":
        return "Status pernikahan/keluarga sesuai profil yang umumnya lolos" if positive \
            else "Status pernikahan/keluarga kurang sesuai profil yang umumnya lolos"
    if group == "sumber":
        h = _hist(rates, "sumber", r["sumber"])
        return f"Sumber informasi lowongan ({r['sumber']}) cenderung menghasilkan pelamar yang lolos{h}" if positive \
            else f"Sumber informasi lowongan ({r['sumber']}) cenderung kurang menghasilkan pelamar yang lolos{h}"
    return group


def explain_rows(contrib: pd.DataFrame, feat: pd.DataFrame, proba: np.ndarray, threshold: float,
                 placement: str = "Cisauk", top_k: int = 3, min_abs: float = 0.08, rates: dict | None = None) -> list[dict]:
    rows = []
    for i, (idx, c) in enumerate(contrib.iterrows()):
        p = float(proba[i])
        lanjut = p >= threshold
        pos = c[c > min_abs].sort_values(ascending=False)
        neg = c[c < -min_abs].sort_values()
        main, other = (pos, neg) if lanjut else (neg, pos)
        r = feat.loc[idx]
        pr = [reason_text(g, g in pos.index, r, placement, rates) for g in main.index[:top_k]]
        co = [reason_text(g, g in pos.index, r, placement, rates) for g in other.index[:2]]
        if not pr:
            pr = ["Tidak ada faktor tunggal yang dominan; skor berada dekat batas keputusan"]
        borderline = abs(p - threshold) <= 0.07
        rows.append(dict(
            rekomendasi="LANJUT INTERVIEW" if lanjut else "TIDAK LANJUT",
            skor=round(p, 3),
            keyakinan="Borderline (disarankan tinjau manual)" if borderline else ("Tinggi" if abs(p - threshold) > 0.25 else "Sedang"),
            alasan_utama=" | ".join(pr),
            faktor_pertimbangan=" | ".join(co),
        ))
    return rows

## 7. Train & evaluate

In [ ]:
import json, sys, warnings
import joblib, numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold

In [ ]:
"""Training + evaluasi.  Pakai:  python train.py --data "Form Loker Tele Marketing (Jawaban) (1).xlsx" """
import argparse, json, sys, warnings
from pathlib import Path
import joblib, numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold


warnings.filterwarnings("ignore")


def lib_versions():
    import importlib.metadata as md
    return {k: md.version(k) for k in ("scikit-learn", "xgboost", "shap", "pandas", "numpy", "rapidfuzz", "joblib")}


def in_windows(ts, windows):
    m = pd.Series(False, index=ts.index)
    for a, b in windows:
        m |= (ts >= pd.Timestamp(a)) & (ts <= pd.Timestamp(b) + pd.Timedelta(days=1))
    return m


def best_threshold(y, p):
    # Youden's J (TPR - FPR): lebih stabil daripada F1 saat prevalensi ~40-50% dan kurva F1 datar
    ts = np.linspace(0.05, 0.95, 91)
    y = np.asarray(y)
    j = [((p >= t) & (y == 1)).sum() / max((y == 1).sum(), 1) - ((p >= t) & (y == 0)).sum() / max((y == 0).sum(), 1) for t in ts]
    return float(ts[int(np.argmax(j))])


def metrics(y, p, thr):
    return dict(auc=roc_auc_score(y, p), ap=average_precision_score(y, p), f1=f1_score(y, p >= thr),
                precision=precision_score(y, p >= thr, zero_division=0), recall=recall_score(y, p >= thr), brier=brier_score_loss(y, p))


def cv_oof(name, X, y, repeats=5):
    oof = np.zeros((repeats, len(y)))
    for r in range(repeats):
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED + r).split(X, y):
            m = make_pipeline(name, SEED).fit(X.iloc[tr], y.iloc[tr])
            oof[r, te] = m.predict_proba(X.iloc[te])[:, 1]
    return oof


def main(a):
    raw = pd.read_excel(a.data)
    if a.rekap:  # label ulang dari Rekap_All (No HP + nama), menggantikan kolom "Lanjut Interview" hasil XLOOKUP nama
        raw["Lanjut Interview"] = relabel(raw, a.rekap).astype(bool)
    geo = GeoResolver()
    feat = dedupe_applicants(build_features(raw, geo, DEFAULT_PLACEMENT))
    rel = feat[in_windows(feat["_ts"], RELIABLE_WINDOWS)].reset_index(drop=True)
    X, y = prepare_X(rel), rel["_y"].astype(int)
    print(f"Pelamar unik total={len(feat)} | dipakai (label andal)={len(rel)} | positif={y.sum()} ({y.mean():.1%})")

    results, oofs = {}, {}
    for name in ("logreg", "rf", "xgb"):
        oof = cv_oof(name, X, y)
        p = oof.mean(0)
        thr = best_threshold(y, p)
        per_rep = [roc_auc_score(y, o) for o in oof]
        results[name] = dict(**metrics(y, p, thr), threshold=thr, auc_sd=float(np.std(per_rep)))
        oofs[name] = p
        print(f"{name:7s} AUC={results[name]['auc']:.3f}±{results[name]['auc_sd']:.3f} AP={results[name]['ap']:.3f} "
              f"F1={results[name]['f1']:.3f} P={results[name]['precision']:.2f} R={results[name]['recall']:.2f} thr={thr:.2f}")
    # baseline: hanya jarak
    from sklearn.linear_model import LogisticRegression
    d = rel[["jarak_km"]].clip(upper=100).fillna(rel["jarak_km"].median())
    pj = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(d, y):
        pj[te] = LogisticRegression().fit(d.iloc[tr], y.iloc[tr]).predict_proba(d.iloc[te])[:, 1]
    results["baseline_jarak_saja"] = dict(auc=roc_auc_score(y, pj), ap=average_precision_score(y, pj))
    print(f"baseline jarak-saja AUC={results['baseline_jarak_saja']['auc']:.3f}  (prevalensi={y.mean():.3f})")

    # uji lintas waktu: model dilatih di RELIABLE_WINDOWS, diuji di CHECK_WINDOW (periode yang tidak dipakai training)
    chk = feat[in_windows(feat["_ts"], [CHECK_WINDOW])].reset_index(drop=True)
    temporal = {}
    if len(chk) >= 20 and chk["_y"].nunique() > 1:
        Xc, yc = prepare_X(chk), chk["_y"].astype(int)
        for name in ("logreg", "rf", "xgb"):
            p = make_pipeline(name, SEED).fit(X, y).predict_proba(Xc)[:, 1]
            temporal[name] = dict(n_test=len(chk), pos_rate=float(yc.mean()), auc=roc_auc_score(yc, p), ap=average_precision_score(yc, p))
            print(f"[periode pembanding {CHECK_WINDOW[0][:7]}..{CHECK_WINDOW[1][:7]}] {name:7s} n={len(chk)} AUC={temporal[name]['auc']:.3f}")
    # pilih model terbaik berdasarkan AP CV (lalu AUC)
    best = max(("logreg", "rf", "xgb"), key=lambda k: (results[k]["ap"] + results[k]["auc"]))
    if a.model:
        best = a.model
    thr = results[best]["threshold"]
    print("Model dipilih:", best, "threshold:", round(thr, 2))

    final = make_pipeline(best, SEED).fit(X, y)
    expl = Explainer(final, background_X=X)
    joblib.dump(dict(pipeline=final, explainer_state=expl.to_state(), versions=lib_versions(), threshold=thr, rates=compute_rates(rel, y), model_name=best, features=FEATURES,
                     train_info=dict(n=len(rel), pos=int(y.sum()), windows=RELIABLE_WINDOWS, placement=DEFAULT_PLACEMENT)), MODEL_PATH)

    # importance global (rata-rata |SHAP| per tema)
    contrib = expl.contributions(X)
    imp = contrib.abs().mean().sort_values(ascending=False).rename("mean_abs_shap")
    imp.to_csv(OUT_DIR / "feature_importance.csv")
    print("\nPengaruh per tema (mean |SHAP|):\n", imp.round(3).to_string())
    out = dict(n_train=len(rel), n_pos=int(y.sum()), cv=results, temporal=temporal, chosen=best, threshold=thr)
    (OUT_DIR / "metrics.json").write_text(json.dumps(out, indent=2, default=float))
    oof = pd.DataFrame(dict(ts=rel["_ts"], y=y, **{k: v for k, v in oofs.items()}))
    oof.to_csv(OUT_DIR / "oof_predictions.csv", index=False)

In [ ]:
from types import SimpleNamespace
main(SimpleNamespace(data=FORM_PATH, rekap=REKAP_PATH, model=None))

### Export for the Streamlit website
The website only needs the trained model file. Download `model_interview.joblib` from `outputs/` and put it in the `models/` folder of the web project. The library versions below must be the ones in the web project's `requirements.txt` (a model saved with one scikit-learn version may not load under another).

In [ ]:
v = joblib.load(MODEL_PATH)["versions"]
req = "\n".join(f"{k}=={val}" for k, val in v.items() if k in ("scikit-learn", "pandas", "numpy", "rapidfuzz", "joblib"))
(OUT_DIR / "model_versions.txt").write_text(req)
print("Model file:", MODEL_PATH)
print("Use these versions in the web project's requirements.txt:\n" + req)

## 8. Accuracy vs coverage
The model is more accurate on applicants it is *sure* about. Raise the confidence margin to trade coverage (share of applicants decided automatically) for accuracy; the rest go to manual review.
Numbers are out-of-fold (applicants the model had not seen).

In [ ]:
import matplotlib.pyplot as plt
oof = pd.read_csv(OUT_DIR / "oof_predictions.csv")
metrics = json.load(open(OUT_DIR / "metrics.json"))
mname, thr = metrics["chosen"], metrics["threshold"]
p, y = oof[mname].values, oof["y"].values
rows = []
for m in [0, .05, .10, .15, .20, .25, .30]:
    decided = (p >= thr + m) | (p < thr - m)
    acc = ((p[decided] >= thr) == y[decided]).mean() if decided.any() else np.nan
    rows.append(dict(margin=m, coverage=decided.mean(), accuracy=acc, n_decided=int(decided.sum())))
tab = pd.DataFrame(rows).round(3)
display(tab)
plt.figure(figsize=(6, 3.5)); plt.plot(tab.coverage, tab.accuracy, "o-"); plt.xlabel("coverage (share decided automatically)")
plt.ylabel("accuracy on decided"); plt.title(f"Accuracy vs coverage ({mname})"); plt.grid(alpha=.3); plt.show()

## 9. Predict (test on the form file)
Writes an Excel file with the original columns plus the AI columns: `AI_Rekomendasi`, `AI_Skor`, `AI_Keyakinan`, `AI_Jarak_km`, `AI_Domisili_Terbaca`, `AI_Alasan_Utama`, `AI_Faktor_Lain`.

In [ ]:
"""Prediksi untuk pelamar baru.

Dipakai sebagai modul:  from predict import Predictor; Predictor().predict(df_form_responses)
Atau CLI uji        :  python predict.py --data pelamar.xlsx --out hasil.xlsx
"""
import argparse
import sys
from pathlib import Path

import joblib
import pandas as pd


OUT_COLS = ["AI_Rekomendasi", "AI_Skor", "AI_Keyakinan", "AI_Jarak_km", "AI_Domisili_Terbaca", "AI_Alasan_Utama", "AI_Faktor_Lain"]


class Predictor:
    def __init__(self, model_path=MODEL_PATH):
        art = joblib.load(model_path)
        self.pipe, self.thr = art["pipeline"], art["threshold"]
        self.expl = Explainer(self.pipe, state=art["explainer_state"])
        self.versions = art.get("versions", {})
        self.info = art["train_info"]
        self.rates = art.get("rates")
        self.geo = GeoResolver()

    def predict(self, raw: pd.DataFrame, placement: str = DEFAULT_PLACEMENT) -> pd.DataFrame:
        raw = raw.reset_index(drop=True)
        feat = build_features(raw, self.geo, placement)
        X = prepare_X(feat)
        proba = self.pipe.predict_proba(X)[:, 1]
        contrib = self.expl.contributions(X)
        place = raw.filter(regex="(?i)lokasi penempatan").iloc[:, 0].astype(str).str.strip().tolist() if raw.filter(regex="(?i)lokasi penempatan").shape[1] else [placement] * len(raw)
        rows = []
        for i in range(len(raw)):
            r = explain_rows(contrib.iloc[[i]], feat.iloc[[i]], proba[[i]], self.thr, place[i] or placement, rates=self.rates)[0]
            rows.append(r)
        res = pd.DataFrame(rows)
        out = pd.DataFrame({
            "AI_Rekomendasi": res["rekomendasi"], "AI_Skor": res["skor"], "AI_Keyakinan": res["keyakinan"],
            "AI_Jarak_km": feat["jarak_km"].round(1), "AI_Domisili_Terbaca": feat["geo_match"],
            "AI_Alasan_Utama": res["alasan_utama"], "AI_Faktor_Lain": res["faktor_pertimbangan"],
        })
        return out

In [ ]:
P = Predictor(MODEL_PATH)
demo = pd.read_excel(FORM_PATH).tail(20)
res = P.predict(demo)
out = pd.concat([demo.reset_index(drop=True), res], axis=1)
out_path = OUT_DIR / "hasil_prediksi_contoh.xlsx"; out.to_excel(out_path, index=False)
display(res[["AI_Rekomendasi", "AI_Skor", "AI_Keyakinan", "AI_Jarak_km", "AI_Domisili_Terbaca"]].head(10))
print("Example reasons for first applicant:\n", res.loc[0, "AI_Alasan_Utama"])
print("Saved:", out_path)

## 10. Write predictions to Google Sheet
Needs a Google service account: enable *Google Sheets API* + *Google Drive API*, create a JSON key, save it in your Drive folder as `service_account.json`, and **share the sheet with the service-account email (Editor)**. Results go to a separate tab **Prediksi AI** (the form tab is never modified). Already-processed applicants are skipped, so re-running is safe.

In [ ]:
"""Sinkronisasi Google Sheet (jawaban Google Form) <-> model.

Alur: baca tab jawaban form -> cari pelamar yang belum diproses -> prediksi -> tulis ke tab terpisah "Prediksi AI".
Tab hasil sengaja DIPISAH dari tab jawaban supaya Google Form tidak bentrok saat menambah kolom/baris.

Contoh:
  python sheet_sync.py --sheet-id <ID_SPREADSHEET> --creds service_account.json            # sekali jalan
  python sheet_sync.py --sheet-id <ID_SPREADSHEET> --creds service_account.json --loop 300 # cek tiap 5 menit
"""
import argparse
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd


OUT_TAB = "Prediksi AI"
HEADER = ["Key", "Timestamp", "Nama Lengkap"] + OUT_COLS + ["Diproses Pada"]


def _col(df: pd.DataFrame, prefix: str) -> pd.Series:
    for c in df.columns:
        if re.sub(r"\s+", " ", str(c)).strip().lower().startswith(prefix):
            return df[c].astype(str)
    return pd.Series([""] * len(df), index=df.index)


def make_keys(df: pd.DataFrame) -> pd.Series:
    ts = _col(df, "timestamp").str.strip()
    ktp = _col(df, "nomor ktp").map(lambda x: re.sub(r"\D", "", x))
    nama = _col(df, "nama lengkap").str.strip().str.lower()
    return ts + "|" + ktp.where(ktp != "", nama)


def pending_rows(form: pd.DataFrame, done_keys: set) -> pd.DataFrame:
    form = form.copy()
    form["_key"] = make_keys(form)
    ok = (_col(form, "timestamp").str.strip() != "") & (_col(form, "nama lengkap").str.strip() != "")
    return form[ok & ~form["_key"].isin(done_keys)]


def build_output_rows(P: Predictor, todo: pd.DataFrame) -> list[list]:
    res = P.predict(todo.drop(columns="_key"))
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows = []
    for k, ts, nm, (_, r) in zip(todo["_key"], _col(todo, "timestamp"), _col(todo, "nama lengkap"), res.iterrows()):
        rows.append([k, ts, nm] + [("" if pd.isna(r[c]) else r[c]) for c in OUT_COLS] + [now])
    return rows


def run_once(sheet_id: str, creds: str, form_tab: str | None, P: Predictor) -> int:
    import gspread

    gc = gspread.service_account(filename=creds)
    sh = gc.open_by_key(sheet_id)
    ws_form = sh.worksheet(form_tab) if form_tab else sh.get_worksheet(0)
    try:
        ws_out = sh.worksheet(OUT_TAB)
    except gspread.WorksheetNotFound:
        ws_out = sh.add_worksheet(OUT_TAB, rows=2000, cols=len(HEADER))
        ws_out.append_row(HEADER)
        ws_out.freeze(rows=1)

    values = ws_form.get_all_values()
    if len(values) < 2:
        return 0
    form = pd.DataFrame(values[1:], columns=values[0])
    done = set(ws_out.col_values(1)[1:])
    todo = pending_rows(form, done)
    if todo.empty:
        return 0
    for i in range(0, len(todo), 50):  # batch kecil supaya aman dari limit API
        ws_out.append_rows(build_output_rows(P, todo.iloc[i:i + 50]), value_input_option="RAW")
    return len(todo)

In [ ]:
if SHEET_ID:
    n = run_once(SHEET_ID, os.path.join(BASE, CREDS_JSON), FORM_TAB, Predictor(MODEL_PATH))
    print("New applicants processed:", n)
else:
    print("SHEET_ID is empty - set it in cell 1 to write to Google Sheets.")